# Data preparation: how `grilla_con_W_hidrica.csv` was built

This notebook documents the pipeline used to construct the 250 m analysis grid from the SRTM DEM and the HydroRIVERS v1.0 river network. It is included for transparency: **the CSV shipped in `data/` was produced by this pipeline** (originally implemented in `original_scripts/CA1.py`).

**You do not need to run this notebook to reproduce the paper.** The CSV is already provided. This notebook is only useful if:

- You want to verify the CSV construction from scratch
- You want to adapt the pipeline to a different study area
- You want to change the analysis resolution (currently 250 m)

**Additional dependencies:** This notebook requires `geopandas`, `rasterio`, and `shapely`, which are already in `requirements.txt` but are only imported here.

**HydroRIVERS v1.0** is not shipped in this repo (large dataset). Download the South America regional shapefile from https://www.hydrosheds.org/products/hydrorivers and place it at `data/HydroRIVERS_v10_sa/HydroRIVERS_v10_sa.shp`.

---

## 1. Read the DEM

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent
DEM_PATH = REPO_ROOT / 'data' / 'DEM_vallebajo.tif'

import rasterio

with rasterio.open(DEM_PATH) as src:
    dem = src.read(1)
    transform = src.transform
    crs = src.crs
    print(f'DEM shape: {dem.shape}')
    print(f'Pixel size (m): {transform.a}, {-transform.e}')
    print(f'CRS: {crs}')
    print(f'Elevation range: {dem.min()} to {dem.max()} m')

## 2. Aggregate to 250 m grid

The DEM is ~30 m; we aggregate to 250 m by mean-pooling.

In [ ]:
CELL_SIZE_M = 250

# Aggregation factor
factor = int(round(CELL_SIZE_M / transform.a))
print(f'Aggregation factor: {factor}x{factor}')

# Crop to a multiple of factor
H = (dem.shape[0] // factor) * factor
W = (dem.shape[1] // factor) * factor
dem_crop = dem[:H, :W]

# Mean-pool
dem_250 = dem_crop.reshape(H // factor, factor, W // factor, factor).mean(axis=(1, 3))
print(f'Aggregated DEM shape: {dem_250.shape}')

## 3. Compute slope

Standard 3×3 spatial derivative (Sobel-like).

In [ ]:
from scipy.ndimage import sobel

dz_dx = sobel(dem_250, axis=1) / (8 * CELL_SIZE_M)  # metres per metre
dz_dy = sobel(dem_250, axis=0) / (8 * CELL_SIZE_M)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

print(f'Slope range: {slope_deg.min():.2f} to {slope_deg.max():.2f} degrees')
print(f'Mean slope: {slope_deg.mean():.2f} degrees')

## 4. Distance to nearest river (HydroRIVERS)

This step requires GeoPandas and the HydroRIVERS shapefile. Skip if you don't have it — the shipped CSV already contains the pre-computed distances.

In [ ]:
RIVERS_PATH = REPO_ROOT / 'data' / 'HydroRIVERS_v10_sa' / 'HydroRIVERS_v10_sa.shp'

if not RIVERS_PATH.exists():
    print(f'HydroRIVERS shapefile not found at {RIVERS_PATH}')
    print('Skipping distance-to-river computation.')
    print('Use the shipped CSV in data/grilla_con_W_hidrica.csv instead.')
else:
    import geopandas as gpd
    from shapely.geometry import box

    # Clip rivers to study extent + 5 km buffer
    with rasterio.open(DEM_PATH) as src:
        bounds = src.bounds
    study_box = box(bounds.left - 5000, bounds.bottom - 5000,
                    bounds.right + 5000, bounds.top + 5000)
    rivers = gpd.read_file(RIVERS_PATH).to_crs(crs)
    rivers_clip = rivers[rivers.intersects(study_box)]
    print(f'River lines within 5 km buffer: {len(rivers_clip)}')

    # For each cell centre in the 250 m grid, compute distance to nearest river
    # This is the main geospatial operation from CA1.py
    # (Placeholder: the actual computation is in CA1.py)
    print('See original_scripts/CA1.py for the full implementation.')

## 5. Compose W_corridor and write CSV

```python
# Simplified logic from CA1.py:

dist_river_norm = (dist_river - dist_river.min()) / (dist_river.max() - dist_river.min())
W_river         = 1 - dist_river_norm

slope_norm      = (slope_deg - slope_deg.min()) / (slope_deg.max() - slope_deg.min())
W_corridor      = W_river * (1 - slope_norm)

df = pd.DataFrame({
    'X': x_coords, 'Y': y_coords,
    'altitud': dem_250.ravel(),
    'pendiente': slope_deg.ravel(),
    'dist_river_m': dist_river.ravel(),
    'W_river': W_river.ravel(),
    'W_corridor': W_corridor.ravel(),
    'presencia_sitio_nuevo': site_flag.ravel(),
})
df.to_csv('grilla_con_W_hidrica.csv', index=False)
```

For the full working implementation, see `original_scripts/CA1.py`.